In [4]:
import pandas as pd
import numpy as np
import os
import sys
import warnings
import re
import unicodedata
import tkinter as tk
from tkinter import filedialog
import xlwings as xw 

# Silenciar advertencias
warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIGURACIÓN
# =============================================================================

HOJA_ADICIONALES = "Adicionales Procesados"
HOJA_MP_CONC = "MP Conciliado"
HOJA_FISERV_CONC = "Fiserv Conciliado"  # <--- CAMBIO: Nombre hoja Fiserv
HOJA_SISTEMA_CONC = "Sistema Conciliado"
# Eliminada la HOJA_GALICIA

COL_TURNO = "TURNO"

MAPA_DIAS = {
    'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles',
    'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}

# --- CLASIFICACIONES ---
TARGET_MP_QR = "QRMERC.PAGO"
# <--- CAMBIO: Renombrado a FISERV y lista de tarjetas
TARGET_FISERV_LISTA = ["MASTER", "VISADEBITO", "VISA", "CABAL", "QRMODO", "MAESTRO", "AMEX", "NARANJA"]
TARGET_EFECTIVO = "EFECTIVO"
TARGET_TRANSFERENCIA = "TRANSFERENCIA"

# =============================================================================
# 2. FUNCIONES DE UTILIDAD
# =============================================================================

def normalizar_texto(texto):
    if pd.isna(texto): return ""
    s = str(texto).lower().strip()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    s = re.sub(r'\s+', ' ', s) 
    return s

def formato_timedelta(td):
    if pd.isna(td): return ""
    total_seconds = int(td.total_seconds())
    signo = ""
    if total_seconds < 0:
        signo = "-"
        total_seconds = abs(total_seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{signo}{hours:02}:{minutes:02}:{seconds:02}"

def cargar_excel(ruta, sheet_name=None):
    try:
        if sheet_name is not None:
            df = pd.read_excel(ruta, sheet_name=sheet_name)
        else:
            df = pd.read_excel(ruta)
        
        # Limpieza de espacios en encabezados
        if not df.empty:
            df.columns = df.columns.astype(str).str.strip()
            
        return df
    except Exception as e:
        print(f"Error cargando {os.path.basename(ruta)}: {e}")
        return pd.DataFrame()

def solicitar_archivo(titulo_ventana):
    print(f"--> Esperando selección de: {titulo_ventana}...")
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    
    ruta = filedialog.askopenfilename(
        title=titulo_ventana,
        filetypes=[("Archivos Excel", "*.xlsx *.xlsm *.xls")]
    )
    root.destroy()
    
    if not ruta:
        print(f"[CANCELADO] No se seleccionó {titulo_ventana}. Saliendo...")
        sys.exit()
    print(f"    Seleccionado: {os.path.basename(ruta)}")
    return ruta

def estandarizar_fecha(df, nombre_origen):
    """Busca columnas de fecha comunes y crea 'datetime_col'"""
    if df.empty: return df
    posibles_nombres = ['datetime_col', 'FECHA', 'Fecha', 'Date', 'date', 'Fecha Creacion', 'Fecha Operacion', 'FECHA_DT']
    col_encontrada = None
    for col in posibles_nombres:
        if col in df.columns:
            col_encontrada = col
            break
    if col_encontrada:
        df['datetime_col'] = pd.to_datetime(df[col_encontrada], dayfirst=True, errors='coerce')
    else:
        # Intento de búsqueda insensible a mayúsculas
        col_lower = {c.lower(): c for c in df.columns}
        if 'fecha' in col_lower:
            df['datetime_col'] = pd.to_datetime(df[col_lower['fecha']], dayfirst=True, errors='coerce')
        else:
            print(f" [ALERTA] No se encontró columna de fecha en {nombre_origen}. Columnas disponibles: {list(df.columns)}")
            df['datetime_col'] = pd.NaT 
    return df

def encontrar_columna(df, opciones):
    """Devuelve el nombre real de la columna si encuentra alguna de las opciones"""
    columnas_lower = {c.lower(): c for c in df.columns}
    for op in opciones:
        if op.lower() in columnas_lower:
            return columnas_lower[op.lower()]
    return None

# =============================================================================
# 3. PROCESAMIENTO PRINCIPAL
# =============================================================================

def generar_arqueo():
    print("=================================================")
    print("      GENERADOR DE ARQUEO - COMPLETO (V5 - FIX FISERV)")
    print("=================================================")

    # 1. Selección de Archivos
    ruta_turnos = solicitar_archivo("1. Selecciona el archivo de TURNOS PROCESADOS")
    ruta_conciliacion = solicitar_archivo("2. Selecciona el Resultado de la CONCILIACIÓN")
    ruta_pagos_manual = solicitar_archivo("3. Selecciona PAGOS SIN REGISTRAR (Manuales)")
    ruta_reporte_caja = solicitar_archivo("4. Selecciona el Reporte de CAJA MAYOR (Ingresos/Egresos)")

    path_macro = solicitar_archivo("5. Selecciona la MACRO destino (Arqueo.xlsm)")

    print("\n>>> Cargando datos en memoria...")
    
    # Carga de hojas
    df_turnos = cargar_excel(ruta_turnos, sheet_name=0)
    df_mp = cargar_excel(ruta_conciliacion, sheet_name=HOJA_MP_CONC)
    
    # --- CARGA FISERV ---
    # Usamos la constante definida arriba (Fiserv Conciliado)
    df_fiserv = cargar_excel(ruta_conciliacion, sheet_name=HOJA_FISERV_CONC)
    
    df_sys_conc = cargar_excel(ruta_conciliacion, sheet_name=HOJA_SISTEMA_CONC)
    df_adicion = cargar_excel(ruta_conciliacion, sheet_name=HOJA_ADICIONALES)
    # df_galicia eliminado
    df_pagos_manual = cargar_excel(ruta_pagos_manual)

    print(">>> Procesando Reporte de Caja...")
    df_caja_extra = cargar_excel(ruta_reporte_caja)
    
    if not df_caja_extra.empty:
        df_caja_extra.columns = df_caja_extra.columns.str.upper().str.strip()
        cols_deseadas = ['FECHA', 'CONCEPTO', 'DETALLE', 'INGRESOS', 'EGRESOS']
        cols_existentes = [c for c in cols_deseadas if c in df_caja_extra.columns]
        df_caja_extra = df_caja_extra[cols_existentes].copy()

    # --- NORMALIZACIÓN ---
    print(">>> Normalizando Datos...")
    
    # A. Turnos y columnas clave
    for df in [df_turnos, df_mp, df_fiserv, df_sys_conc, df_adicion, df_pagos_manual]:
        c_turno = encontrar_columna(df, ['TURNO', 'Turno'])
        if c_turno:
            df.rename(columns={c_turno: COL_TURNO}, inplace=True)
            df[COL_TURNO] = df[COL_TURNO].replace(r'^\s*$', pd.NA, regex=True).ffill()
            df[COL_TURNO] = df[COL_TURNO].astype(str).str.upper().str.strip().str.replace('.0', '')
        elif not df.empty and df is df_turnos:
            print(f"[ERROR CRITICO] No se encontró columna TURNO en el archivo de Turnos.")
            return

    # B. Fechas de Turnos
    col_f_ap = encontrar_columna(df_turnos, ['Fecha Apertura', 'FECHA APERTURA', 'Fecha Ap', 'F. Apertura'])
    col_h_ap = encontrar_columna(df_turnos, ['Hs Ap. Caja', 'Hora Apertura', 'Hs Apertura', 'Hora Ap', 'H Apertura'])
    col_f_ci = encontrar_columna(df_turnos, ['Fecha Cierre', 'FECHA CIERRE', 'Fecha Ci', 'F. Cierre'])
    col_h_ci = encontrar_columna(df_turnos, ['Hs Cierre Caja', 'Hora Cierre', 'Hs Cierre', 'Hora Ci', 'H Cierre'])

    if not all([col_f_ap, col_h_ap, col_f_ci, col_h_ci]):
        print("\n[ERROR] No se encuentran columnas de Fecha/Hora en Turnos.")
        return 

    try:
        df_turnos['Apertura'] = pd.to_datetime(pd.to_datetime(df_turnos[col_f_ap], dayfirst=True).dt.strftime('%Y-%m-%d') + ' ' + df_turnos[col_h_ap].astype(str), errors='coerce')
        df_turnos['Cierre'] = pd.to_datetime(pd.to_datetime(df_turnos[col_f_ci], dayfirst=True).dt.strftime('%Y-%m-%d') + ' ' + df_turnos[col_h_ci].astype(str), errors='coerce')
    except Exception as e:
        df_turnos['Apertura'] = pd.to_datetime(df_turnos[col_f_ap], dayfirst=True, errors='coerce')
        df_turnos['Cierre'] = pd.to_datetime(df_turnos[col_f_ci], dayfirst=True, errors='coerce')

    df_turnos['Fecha_Dia'] = pd.to_datetime(df_turnos[col_f_ap], dayfirst=True).dt.date
    col_ingresos_extras = encontrar_columna(df_turnos, ['ingresos extras', 'ingreso extra', 'otros ingresos'])

    # C. Datos Transaccionales
    df_mp = estandarizar_fecha(df_mp, "MercadoPago")
    df_fiserv = estandarizar_fecha(df_fiserv, "Fiserv")
    df_sys_conc = estandarizar_fecha(df_sys_conc, "Sistema")
    
    # Asegurar que monto_col_numeric sea número en todos
    for df in [df_mp, df_fiserv, df_sys_conc]:
        # Si no existe monto_col_numeric, lo buscamos
        if 'monto_col_numeric' not in df.columns:
            col_importe = encontrar_columna(df, ['IMPORTE', 'Monto', 'Amount', 'Monto Total', 'Total'])
            if col_importe:
                 df['monto_col_numeric'] = pd.to_numeric(df[col_importe], errors='coerce').fillna(0)
            else:
                 df['monto_col_numeric'] = 0.0
        else:
            # Si existe, aseguramos que sea float
            df['monto_col_numeric'] = pd.to_numeric(df['monto_col_numeric'], errors='coerce').fillna(0)
        
        col_id_orig = encontrar_columna(df, ['ID_Original', 'ID Original'])
        if col_id_orig: df['ID_Original'] = df[col_id_orig].astype(str).str.replace(r'\.0$', '', regex=True)
            
        col_id_cruce = encontrar_columna(df, ['ID_Cruce', 'ID Cruce'])
        if col_id_cruce: df['ID_Cruce'] = df[col_id_cruce].astype(str).str.replace(r'\.0$', '', regex=True)

    # D. Adicionales
    if 'FECHA_DT' not in df_adicion.columns: df_adicion['FECHA_DT'] = pd.Series(dtype='object')
    if COL_TURNO not in df_adicion.columns: df_adicion[COL_TURNO] = pd.Series(dtype='object')
        
    if not df_adicion.empty:
        col_detalle = encontrar_columna(df_adicion, ['DETALLE', 'Concepto'])
        col_tipo = encontrar_columna(df_adicion, ['TIPO', 'Tipo Movimiento'])
        col_imp_adic = encontrar_columna(df_adicion, ['IMPORTE', 'Monto'])
        
        if col_detalle: df_adicion['DETALLE_NORM'] = df_adicion[col_detalle].apply(normalizar_texto)
        else: df_adicion['DETALLE_NORM'] = ""
        if col_tipo: df_adicion['TIPO_NORM'] = df_adicion[col_tipo].apply(normalizar_texto)
        else: df_adicion['TIPO_NORM'] = ""
        if col_imp_adic: df_adicion['IMPORTE'] = pd.to_numeric(df_adicion[col_imp_adic], errors='coerce').fillna(0)
        
        col_recuento = encontrar_columna(df_adicion, ['Total Recuento', 'Recuento'])
        df_adicion['Total Recuento'] = pd.to_numeric(df_adicion[col_recuento], errors='coerce').fillna(0) if col_recuento else 0
        col_fecha_adic = encontrar_columna(df_adicion, ['FECHA', 'Fecha'])
        if col_fecha_adic: df_adicion['FECHA_DT'] = pd.to_datetime(df_adicion[col_fecha_adic], dayfirst=True, errors='coerce').dt.date
    else:
        df_adicion['DETALLE_NORM'] = ""; df_adicion['TIPO_NORM'] = ""
        df_adicion['IMPORTE'] = 0.0; df_adicion['Total Recuento'] = 0.0

    # E. Pagos Manuales
    if not df_pagos_manual.empty:
        col_pm_fecha = encontrar_columna(df_pagos_manual, ['FECHA', 'Fecha', 'Date'])
        if col_pm_fecha: df_pagos_manual['FECHA_DT'] = pd.to_datetime(df_pagos_manual[col_pm_fecha], dayfirst=True, errors='coerce').dt.date
        else: df_pagos_manual['FECHA_DT'] = None
        col_pm_imp = encontrar_columna(df_pagos_manual, ['IMPORTE', 'Monto', 'Valor'])
        if col_pm_imp: df_pagos_manual['IMPORTE'] = pd.to_numeric(df_pagos_manual[col_pm_imp], errors='coerce').fillna(0)
        else: df_pagos_manual['IMPORTE'] = 0.0
    else:
        df_pagos_manual['FECHA_DT'] = None; df_pagos_manual['IMPORTE'] = 0.0

    # F. Sistema
    if not df_sys_conc.empty:
        col_medio = encontrar_columna(df_sys_conc, ['Medio_Cobro_Norm', 'Medio Cobro'])
        if col_medio: df_sys_conc['Medio_Cobro_Norm'] = df_sys_conc[col_medio].astype(str).str.strip().str.upper()
        col_cub = encontrar_columna(df_sys_conc, ['CUB', 'Cubiertos', 'Pax'])
        if col_cub: df_sys_conc['CUB'] = pd.to_numeric(df_sys_conc[col_cub], errors='coerce').fillna(0)
        else: df_sys_conc['CUB'] = 0

    # --- CÁLCULO ---
    print(">>> Calculando por Rango Horario...")
    resultados = []
    cols_horas = [f"{h:02d}:00 - {h+1:02d}:00" for h in range(24)]

    for idx, row in df_turnos.iterrows():
        turno = row[COL_TURNO]
        fecha_dia = row['Fecha_Dia']
        inicio = row['Apertura']
        fin = row['Cierre']
        if pd.isna(inicio) or pd.isna(fin): continue

        dia_ingles = pd.to_datetime(row[col_f_ap]).strftime('%A')
        dia_espanol = MAPA_DIAS.get(dia_ingles, dia_ingles)
        col_encargado = encontrar_columna(df_turnos, ['Encargado', 'Cajero', 'Responsable'])
        encargado = row[col_encargado] if col_encargado else ''

        val_ingresos_extras = 0.0
        if col_ingresos_extras:
            val = row[col_ingresos_extras]
            val_ingresos_extras = float(val) if pd.notna(val) and str(val).replace('.','').isdigit() else 0.0

        # Filtros de Tiempo y Turno
        mask_mp = df_mp['datetime_col'].between(inicio, fin)
        mask_fiserv = df_fiserv['datetime_col'].between(inicio, fin)
        
        margen = pd.Timedelta(hours=12)
        mask_sys = (df_sys_conc[COL_TURNO] == turno) & (df_sys_conc['datetime_col'].between(inicio - margen, fin + margen))
        mask_adicion = (df_adicion['FECHA_DT'] == fecha_dia) & (df_adicion[COL_TURNO] == turno)
        mask_pagos_man = (df_pagos_manual['FECHA_DT'] == fecha_dia) & (df_pagos_manual[COL_TURNO] == turno)
        
        # DataFrames Filtrados
        d_sys = df_sys_conc[mask_sys].copy()
        d_mp = df_mp[mask_mp].copy()
        d_fiserv = df_fiserv[mask_fiserv].copy()
        d_adic = df_adicion[mask_adicion].copy()
        d_pagos_man = df_pagos_manual[mask_pagos_man].copy()

        monto_pagos_manuales = d_pagos_man['IMPORTE'].sum()
        obj_horas_trabajadas = (fin - inicio)
        
        # Datos de Horas
        hs_primera_venta = pd.NaT; hs_ultima_venta = pd.NaT
        str_primera_venta = ""; str_ultima_venta = ""; str_ap_vs_1er = ""; str_ult_vs_cierre = ""; str_intervalo = ""

        if not d_sys.empty:
            hs_primera_venta = d_sys['datetime_col'].min()
            hs_ultima_venta = d_sys['datetime_col'].max()
            if pd.notna(hs_primera_venta):
                str_primera_venta = hs_primera_venta.strftime('%H:%M:%S')
                str_ap_vs_1er = formato_timedelta(hs_primera_venta - inicio)
            if pd.notna(hs_ultima_venta):
                str_ultima_venta = hs_ultima_venta.strftime('%H:%M:%S')
                str_ult_vs_cierre = formato_timedelta(fin - hs_ultima_venta)
            if pd.notna(hs_primera_venta) and pd.notna(hs_ultima_venta):
                str_intervalo = formato_timedelta(hs_ultima_venta - hs_primera_venta)

        # Clasificación de Sistema
        if 'Medio_Cobro_Norm' in d_sys.columns:
            d_sys_mp = d_sys[d_sys['Medio_Cobro_Norm'] == TARGET_MP_QR]
            d_sys_fiserv = d_sys[d_sys['Medio_Cobro_Norm'].isin(TARGET_FISERV_LISTA)]
        else:
            d_sys_mp = pd.DataFrame(columns=d_sys.columns)
            d_sys_fiserv = pd.DataFrame(columns=d_sys.columns)
        
        # Ajuste de Falsos Positivos
        ids_mp = set(d_mp[d_mp['Estado'] == 'Conciliado']['ID_Cruce'].dropna().unique())
        ids_fiserv = set(d_fiserv[d_fiserv['Estado'] == 'Conciliado']['ID_Cruce'].dropna().unique())
        
        mask_falso_mp = d_sys_mp['ID_Original'].isin(ids_fiserv)
        mask_falso_fiserv = d_sys_fiserv['ID_Original'].isin(ids_mp)
        
        monto_falso_mp = d_sys_mp[mask_falso_mp]['monto_col_numeric'].sum()
        monto_falso_fiserv = d_sys_fiserv[mask_falso_fiserv]['monto_col_numeric'].sum()
        
        mp_sys_base = d_sys_mp['monto_col_numeric'].sum()
        fiserv_sys_base = d_sys_fiserv['monto_col_numeric'].sum()

        mp_sys_total_ajustado = mp_sys_base - monto_falso_mp + monto_falso_fiserv
        fiserv_sys_total_ajustado = fiserv_sys_base - monto_falso_fiserv + monto_falso_mp

        # Efectivo
        monto_recaudacion = d_adic[d_adic['DETALLE_NORM'].str.contains('recaudacion', na=False)]['IMPORTE'].sum()
        efectivo_inicial = d_adic[d_adic['DETALLE_NORM'].str.contains('apertura saldo caja anterior', na=False)]['IMPORTE'].sum()
        monto_apertura_total = d_adic[d_adic['DETALLE_NORM'].str.contains('apertura', na=False)]['IMPORTE'].sum()
        
        frases_excluidas = ['apertura saldo caja anterior', 'ajuste por recuento', 'correccion caja anterior']
        cond_detalle_valido = ~d_adic['DETALLE_NORM'].str.contains('|'.join(frases_excluidas), regex=True, na=False)
        monto_pagos_sistema = abs(d_adic[(d_adic['TIPO_NORM'] == 'egresos') & cond_detalle_valido]['IMPORTE'].sum())
        
        total_pagos_consolidado = monto_pagos_sistema + monto_pagos_manuales
        recuento_sistema = monto_apertura_total + monto_recaudacion - total_pagos_consolidado
        dif_efectivo = d_adic['Total Recuento'].sum() - recuento_sistema
        
        # TOTALES REALES (USANDO LA COLUMNA NUMÉRICA)
        mp_real = d_mp['monto_col_numeric'].sum()
        # >>> CORRECCIÓN CLAVE AQUÍ <<<
        fiserv_real = d_fiserv['monto_col_numeric'].sum() 

        # Cta Cte y Transferencias
        mask_cc = pd.Series(False, index=d_sys.index)
        if 'Medio_Cobro_Norm' in d_sys.columns:
            mask_cc = d_sys['Medio_Cobro_Norm'].str.contains('CTA', na=False) | d_sys['Medio_Cobro_Norm'].str.contains('CUENTA', na=False)
        
        ctacte_sys = d_sys[mask_cc]['monto_col_numeric'].sum() if 'Medio_Cobro_Norm' in d_sys.columns else 0
        conciliado_ctacte_sys = d_sys[mask_cc & (d_sys['Estado'] == 'Conciliado')]['monto_col_numeric'].sum() if 'Medio_Cobro_Norm' in d_sys.columns else 0
        ctacte_real = ctacte_sys - conciliado_ctacte_sys

        transferencias_sys = d_sys[d_sys['Medio_Cobro_Norm'].str.contains(TARGET_TRANSFERENCIA, na=False)]['monto_col_numeric'].sum() if 'Medio_Cobro_Norm' in d_sys.columns else 0
        
        ventas_facturadas = 0
        if not d_sys.empty:
            col_comp = encontrar_columna(d_sys, ['COMPROBANT', 'COMPROBANTE', 'TIPO', 'TIPO COMP'])
            if col_comp:
                series_comp = d_sys[col_comp].astype(str).str.replace(' ', '').str.upper()
                prefijos = ('FCA0008', 'FCB0008', 'NCA0008', 'NCB0008')
                mask_fact = series_comp.str.startswith(prefijos)
                ventas_facturadas = d_sys[mask_fact]['monto_col_numeric'].sum()

        conciliado_efectivo_sys = d_sys[(d_sys['Medio_Cobro_Norm'] == TARGET_EFECTIVO) & (d_sys['Estado']=='Conciliado')]['monto_col_numeric'].sum() if 'Medio_Cobro_Norm' in d_sys.columns else 0
        efectivo_real_calculado = monto_recaudacion - conciliado_efectivo_sys

        ventas_totales = mp_real + fiserv_real + efectivo_real_calculado + ctacte_sys + transferencias_sys
        cant_comensales = d_sys['CUB'].sum()
        cant_tickets = d_sys['ID_Original'].nunique() if not d_sys.empty else 0
        ticket_por_comensal = ventas_totales / cant_comensales if cant_comensales > 0 else 0
        ticket_promedio = ventas_totales / cant_tickets if cant_tickets > 0 else 0

        # Diferencias
        diferencia_mp = mp_real - mp_sys_total_ajustado
        diferencia_fiserv = fiserv_real - fiserv_sys_total_ajustado
        diferencia_facturacion = ventas_facturadas - ventas_totales
        
        pct_diferencia_mp = (diferencia_mp / mp_real) if mp_real != 0 else 0.0
        pct_diferencia_fiserv = (diferencia_fiserv / fiserv_real) if fiserv_real != 0 else 0.0
        pct_diferencia_efectivo = (dif_efectivo / efectivo_real_calculado) if efectivo_real_calculado != 0 else 0.0
        pct_diferencia_facturacion = (diferencia_facturacion / ventas_totales) if ventas_totales != 0 else 0.0

        r = {
            'Fecha': row[col_f_ap], 'Dia': dia_espanol, 'TURNO': turno, 'Encargado': encargado,
            'Apertura': inicio, 'Hora Apertura': inicio, 'Cierre': fin, 'Hora Cierre': fin,
            'Horas Trabajadas': formato_timedelta(obj_horas_trabajadas),
            'Hs Primera Venta Sistema': str_primera_venta, 'Hs Ap Vs 1er venta': str_ap_vs_1er,
            'Hs Ultima Venta Sistema': str_ultima_venta, 'Hs Ultima vta Vs cierre': str_ult_vs_cierre,
            'Intervalo ventas': str_intervalo,
            'Ventas Totales': ventas_totales,
            'Cantidad de comensales': cant_comensales, 'Cantidad de ticket': cant_tickets,
            'Ticket por comensal promedio': ticket_por_comensal, 'Ticket promedio': ticket_promedio,
            'Ventas MP Real': mp_real, 'Ventas Sistema MP Real': mp_sys_base, 'Ventas Sistema MP Correcto': mp_sys_total_ajustado,
            'Conciliado MP REAL': d_mp[d_mp['Estado']=='Conciliado']['monto_col_numeric'].sum(),
            'No conciliado MP REAL': d_mp[d_mp['Estado']!='Conciliado']['monto_col_numeric'].sum(),
            'Conciliado Sistema MP': d_sys_mp[d_sys_mp['Estado']=='Conciliado']['monto_col_numeric'].sum(),
            'No conciiado Sistema MP': d_sys_mp[d_sys_mp['Estado']!='Conciliado']['monto_col_numeric'].sum(),
            'Diferencia MP sistema vs real': diferencia_mp,
            '% Diferencia MP': pct_diferencia_mp, 
            'Estado Dif MP': "Sobrante" if diferencia_mp > 0 else "Faltante" if diferencia_mp < 0 else "OK",
            'Efectivo Inicial': efectivo_inicial, 'Efectivo Total': monto_recaudacion,
            'Conciliado Efectivo, Sistema': conciliado_efectivo_sys, 'Efectivo Real': efectivo_real_calculado,
            'Pagos': monto_pagos_sistema, 'Pagos en efectivo manual': monto_pagos_manuales,
            'Recuento Efectivo': d_adic['Total Recuento'].sum(), 'Recuento sistema efectivo': recuento_sistema,
            'Diferencia Efectivo': dif_efectivo,
            '% Diferencia Efectivo': pct_diferencia_efectivo, 
            'Estado Diferencia Efectivo': "Sobrante" if dif_efectivo > 0 else "Faltante" if dif_efectivo < 0 else "OK",
            
            # --- DATOS FISERV ---
            'Ventas FISERV Real': fiserv_real, 
            'Ventas sistema Real': fiserv_sys_base, 
            'Ventas sistema Fiserv Correcto': fiserv_sys_total_ajustado,
            'Conciliado FISERV REAL': d_fiserv[d_fiserv['Estado']=='Conciliado']['monto_col_numeric'].sum(),
            'No conciliado FISERV REAL': d_fiserv[d_fiserv['Estado']!='Conciliado']['monto_col_numeric'].sum(),
            'Conciliado Sistema FISERV': d_sys_fiserv[d_sys_fiserv['Estado']=='Conciliado']['monto_col_numeric'].sum(),
            'No conciliado Sistema FISERV': d_sys_fiserv[d_sys_fiserv['Estado']!='Conciliado']['monto_col_numeric'].sum(),
            'Diferencia FISERV sistema vs real': diferencia_fiserv,
            '% Diferencia FISERV': pct_diferencia_fiserv, 
            'Estado Dif FISERV': "Sobrante" if diferencia_fiserv > 0 else "Faltante" if diferencia_fiserv < 0 else "OK",
            
            'Ingresos Extras': val_ingresos_extras, 'Transferencias': transferencias_sys,
            'Cta Cte Total': ctacte_sys, 'Conciliado Cta Cte': conciliado_ctacte_sys, 'Cta Cte Real': ctacte_real,
            'Ventas facturadas': ventas_facturadas,
            'Ideal facturacion': ventas_totales, 'Diferencia de facturacion': diferencia_facturacion,
            '% Diferencia facturacion': pct_diferencia_facturacion 
        }
        for h in cols_horas: r[h] = 0.0
        if not d_sys.empty and 'datetime_col' in d_sys.columns:
            d_sys_h = d_sys.copy()
            d_sys_h['h_str'] = d_sys_h['datetime_col'].apply(lambda x: f"{x.hour:02d}:00 - {x.hour+1:02d}:00")
            for k, v in d_sys_h.groupby('h_str')['monto_col_numeric'].sum().to_dict().items():
                if k in r: r[k] = v
        resultados.append(r)

    df_final = pd.DataFrame(resultados)
    
    col_order = [
        'Fecha', 'Dia', 'TURNO', 'Encargado', 
        'Apertura', 'Hora Apertura', 'Cierre', 'Hora Cierre', 'Horas Trabajadas',
        'Hs Primera Venta Sistema', 'Hs Ap Vs 1er venta', 'Hs Ultima Venta Sistema', 'Hs Ultima vta Vs cierre', 'Intervalo ventas',
        'Ventas Totales', 'Cantidad de comensales', 'Cantidad de ticket', 'Ticket por comensal promedio', 'Ticket promedio',
        'Ventas MP Real', 'Ventas Sistema MP Real', 'Ventas Sistema MP Correcto', 'Conciliado MP REAL', 'No conciliado MP REAL', 
        'Conciliado Sistema MP', 'No conciiado Sistema MP', 
        'Diferencia MP sistema vs real', '% Diferencia MP', 'Estado Dif MP',
        'Efectivo Inicial', 'Efectivo Total', 'Conciliado Efectivo, Sistema', 'Efectivo Real', 
        'Pagos', 'Pagos en efectivo manual', 
        'Recuento Efectivo', 'Recuento sistema efectivo', 
        'Diferencia Efectivo', '% Diferencia Efectivo', 'Estado Diferencia Efectivo',
        
        # --- COLUMNAS FISERV ---
        'Ventas FISERV Real', 'Ventas sistema Real', 'Ventas sistema Fiserv Correcto', 'Conciliado FISERV REAL', 'No conciliado FISERV REAL', 
        'Conciliado Sistema FISERV', 'No conciliado Sistema FISERV', 
        'Diferencia FISERV sistema vs real', '% Diferencia FISERV', 'Estado Dif FISERV',
        
        'Ingresos Extras', 'Transferencias',
        'Cta Cte Total', 'Conciliado Cta Cte', 'Cta Cte Real', 
        'Ventas facturadas', 'Ideal facturacion', 
        'Diferencia de facturacion', '% Diferencia facturacion' 
    ] + cols_horas

    for c in col_order: 
        if c not in df_final.columns: df_final[c] = 0
    df_final = df_final[col_order]

    # =========================================================================
    # 5. GUARDADO EN EL MACRO (XLWINGS)
    # =========================================================================
    print(f"\n>>> Exportando a MACRO: {path_macro}")

    try:
        app = xw.App(visible=False)
        wb = app.books.open(path_macro)
        
        # --- HOJA 1: ARQUEO POR TURNO (Calculado) ---
        nombre_hoja = 'Arqueo_por_Turno'
        try:
            ws = wb.sheets[nombre_hoja]
            ws.cells.clear() 
        except:
            ws = wb.sheets.add(nombre_hoja)

        if not df_final.empty:
            ws.range('A1').options(index=False).value = df_final
            ws.autofit()
            
            # Formatos Hoja 1
            cols_no_moneda = [
                'Fecha', 'Dia', 'TURNO', 'Encargado', 
                'Hs Primera Venta Sistema', 'Hs Ap Vs 1er venta', 'Hs Ultima Venta Sistema', 
                'Hs Ultima vta Vs cierre', 'Intervalo ventas', 'Horas Trabajadas',
                'Estado Dif MP', 'Estado Diferencia Efectivo', 'Estado Dif FISERV',
                'Cantidad de comensales', 'Cantidad de ticket'
            ]
            cols_porcentaje = [
                '% Diferencia MP', '% Diferencia Efectivo', '% Diferencia FISERV',
                '% Diferencia facturacion'
            ]
            cols_datetime_full = ['Apertura', 'Cierre']
            cols_time_only = ['Hora Apertura', 'Hora Cierre']
            last_row = len(df_final) + 1
            
            for i, col_name in enumerate(df_final.columns):
                rango = ws.range((2, i+1), (last_row, i+1))
                
                if col_name in cols_porcentaje:
                    rango.number_format = '0,00%' 
                elif col_name in cols_datetime_full:
                    rango.number_format = 'dd/mm/yyyy hh:mm'
                    rango.api.HorizontalAlignment = -4108 
                elif col_name in cols_time_only:
                    rango.number_format = 'hh:mm:ss'
                    rango.api.HorizontalAlignment = -4108 
                elif col_name in cols_no_moneda:
                    rango.api.HorizontalAlignment = -4108 
                else:
                    rango.api.NumberFormatLocal = '#.##0,00'
        else:
            ws.range('A1').value = "Sin datos"

        # --- HOJA 2: PAGOS SIN REGISTRAR ---
        nombre_hoja_pagos = 'Pagos sin registrar'
        try:
            ws_pagos = wb.sheets[nombre_hoja_pagos]
            ws_pagos.cells.clear()
        except:
            ws_pagos = wb.sheets.add(nombre_hoja_pagos)
            
        if not df_pagos_manual.empty:
            df_export_pagos = df_pagos_manual.copy()
            if 'FECHA_DT' in df_export_pagos.columns:
                df_export_pagos.drop(columns=['FECHA_DT'], inplace=True)
            ws_pagos.range('A1').options(index=False).value = df_export_pagos
            ws_pagos.autofit()
        else:
            ws_pagos.range('A1').value = "No hay pagos manuales registrados."

        # --- HOJA 3: MOVIMIENTOS CAJA ---
        nombre_hoja_caja = 'Movimientos Caja'
        try:
            ws_caja = wb.sheets[nombre_hoja_caja]
            ws_caja.cells.clear()
        except:
            ws_caja = wb.sheets.add(nombre_hoja_caja)
            
        if not df_caja_extra.empty:
            ws_caja.range('A1').options(index=False).value = df_caja_extra
            ws_caja.autofit()
        else:
            ws_caja.range('A1').value = "Sin datos de caja."

        wb.save()
        print(f"    -> Exportación exitosa en: {os.path.basename(path_macro)}")

    except Exception as e:
        print(f"    [ERROR] xlwings: {e}")
        import traceback; traceback.print_exc()
    finally:
        try: wb.close(); app.quit()
        except: pass

    print("¡Proceso Finalizado!")
if __name__ == "__main__":
    generar_arqueo()
    input("\nPresiona ENTER para cerrar la ventana...")

      GENERADOR DE ARQUEO - COMPLETO (V5 - FIX FISERV)
--> Esperando selección de: 1. Selecciona el archivo de TURNOS PROCESADOS...
    Seleccionado: Turnos_Procesados.xlsx
--> Esperando selección de: 2. Selecciona el Resultado de la CONCILIACIÓN...
    Seleccionado: Resultado_Conciliacion.xlsx
--> Esperando selección de: 3. Selecciona PAGOS SIN REGISTRAR (Manuales)...
    Seleccionado: Pagos sin registrar.xlsx
--> Esperando selección de: 4. Selecciona el Reporte de CAJA MAYOR (Ingresos/Egresos)...
    Seleccionado: MAYOR 9-02 (1).XLSX
--> Esperando selección de: 5. Selecciona la MACRO destino (Arqueo.xlsm)...
    Seleccionado: ARQUEO INVERNADAS PAULA.xlsm

>>> Cargando datos en memoria...
>>> Procesando Reporte de Caja...
>>> Normalizando Datos...
>>> Calculando por Rango Horario...

>>> Exportando a MACRO: G:/Mi unidad/Modelo de automatizaciones/Las Invernadas Paula/ARQUEO INVERNADAS PAULA.xlsm
    -> Exportación exitosa en: ARQUEO INVERNADAS PAULA.xlsm
¡Proceso Finalizado!
